# Setup, inspect what title/reference metadata each source actually has

In [1]:
import sys, os
from pathlib import Path
import logging

def find_project_root(marker="backend", start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not find a '{marker}' folder above {current}")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "scripts"))

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)

from medrag.processing.storage import load_chunks
from run_chunking import WHO_TOPIC_GROUPS

CHUNKS_DIR = str(PROJECT_ROOT / "data" / "processed" / "chunks")

# one real chunk per source, inspect full metadata
who_chunk = next(iter(load_chunks(source="who", topic="hypertension", output_dir=CHUNKS_DIR)))
openfda_chunk = next(iter(load_chunks(source="openfda", topic="diabetes", output_dir=CHUNKS_DIR)))
pubmed_chunk = next(iter(load_chunks(source="pubmed", topic="diabetes", output_dir=CHUNKS_DIR)))

for label, chunk in [("WHO", who_chunk), ("OpenFDA", openfda_chunk), ("PubMed", pubmed_chunk)]:
    print(f"=== {label} ===")
    print(f"  chunk_id: {chunk.chunk_id}")
    print(f"  source_id: {chunk.source_id}")
    print(f"  metadata: {chunk.metadata}")
    print()

=== WHO ===
  chunk_id: hypertension_who_text_0
  source_id: hypertension
  metadata: {'title': 'Guideline for the pharmacological treatment of hypertension in adults'}

=== OpenFDA ===
  chunk_id: Glimepiride_openfda_0
  source_id: Glimepiride
  metadata: {'field': 'indications_and_usage'}

=== PubMed ===
  chunk_id: 42480168_pubmed_0
  source_id: 42480168
  metadata: {'title': 'Determinants and promotion strategies for type 1 diabetes screening in children: A qualitative study from a parental perspective.'}



In [ ]:
print(who_chunk.metadata.keys())

dict_keys(['title'])


# Check whether source_url is accessible another way right now

In [ ]:
import json

WHO_RAW_DIR = str(PROJECT_ROOT / "data" / "raw" / "who")  # adjust if your Phase 4 save path differs
print(Path(WHO_RAW_DIR).exists())
if Path(WHO_RAW_DIR).exists():
    print(list(Path(WHO_RAW_DIR).glob("*"))[:5])

True
[WindowsPath('C:/Users/DELL/Desktop/medrag/data/raw/who/anemia_in_pregnancy.json'), WindowsPath('C:/Users/DELL/Desktop/medrag/data/raw/who/anxiety_disorder.json'), WindowsPath('C:/Users/DELL/Desktop/medrag/data/raw/who/asthma.json'), WindowsPath('C:/Users/DELL/Desktop/medrag/data/raw/who/breast_cancer.json'), WindowsPath('C:/Users/DELL/Desktop/medrag/data/raw/who/copd.json')]


# Build a source_id → source_url lookup from raw WHO guideline files

In [4]:
def build_who_source_url_lookup(who_raw_dir: str) -> dict:
    """Map WHO source_id (topic slug, matches Chunk.source_id) -> source_url,
    read directly from Phase 4's saved raw Guideline JSON files - no need
    to touch the chunker to get this, since the URL already exists on
    disk from ingestion."""
    lookup = {}
    for filepath in Path(who_raw_dir).glob("*.json"):
        with open(filepath, encoding="utf-8") as f:
            data = json.load(f)
        topic = filepath.stem
        lookup[topic] = data.get("source_url")
    return lookup

who_source_urls = build_who_source_url_lookup(WHO_RAW_DIR)
print(f"Loaded {len(who_source_urls)} WHO source URLs")

# spot check
print(who_source_urls.get("hypertension"))

Loaded 24 WHO source URLs
https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content


# Build the citation resolution function

In [5]:
import re

def get_display_info(payload: dict, who_source_urls: dict) -> dict:
    """Return {title, url} for a chunk's payload, using per-source
    metadata discovered in Cell 1/2/3. url may be None if no direct
    link is available for this source/case."""
    source = payload["source"]

    if source == "who":
        title = payload["metadata"].get("title", payload.get("source_id", "WHO Guideline"))
        url = who_source_urls.get(payload["source_id"])
        return {"title": title, "url": url}

    if source == "openfda":
        drug_name = payload["source_id"]
        field = payload["metadata"].get("field", "")
        field_display = field.replace("_", " ").title()
        title = f"{drug_name} — FDA Label ({field_display})" if field else f"{drug_name} — FDA Label"
        return {"title": title, "url": None}  # no stable public OpenFDA per-drug URL captured at ingestion

    if source == "pubmed":
        title = payload["metadata"].get("title", f"PubMed article {payload['source_id']}")
        pmid = payload["source_id"]
        url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
        return {"title": title, "url": url}

    return {"title": "Unknown source", "url": None}


def extract_used_citation_numbers(answer_text: str) -> set:
    """Find every bracketed number actually used in the generated
    answer text, e.g. '[1]' or '[1][3]' -> {1, 3}."""
    return {int(n) for n in re.findall(r"\[(\d+)\]", answer_text)}


def build_citations(answer_text: str, results: list, who_source_urls: dict) -> list:
    """Resolve every citation marker actually used in the answer back
    to a structured citation object. results is the same reranked list
    used to build the numbered context block - index N-1 corresponds
    to marker [N]."""
    used_numbers = extract_used_citation_numbers(answer_text)
    citations = []
    for n in sorted(used_numbers):
        if n < 1 or n > len(results):
            continue  # model referenced a number outside the actual context block range
        result = results[n - 1]
        payload = result["payload"]
        display = get_display_info(payload, who_source_urls)
        citations.append({
            "marker": n,
            "chunk_id": payload["chunk_id"],
            "source": payload["source"],
            "title": display["title"],
            "url": display["url"],
            "linked_images": payload.get("linked_images", []),
        })
    return citations

In [9]:
from medrag.embeddings.qdrant_client import get_qdrant_client
from medrag.retrieval.reranking import search_with_reranking
from config.settings import settings

qdrant = get_qdrant_client(settings.qdrant_url or "http://localhost:6333")

query = "side effects of ACE inhibitors"
results = search_with_reranking(qdrant, query, candidate_pool_size=20, top_n=5)

# reuse a generated answer from Phase 14, or regenerate one now
answer_text = "The major adverse effects of ACE inhibitors include dry cough and renal dysfunction in patients with impaired renal function [1]. Additionally, ACE inhibitors have been associated with angioedema, hypovolemia, hypotension, hyperkalemia, and, more rarely, cholestatic jaundice, hepatic failure, neutropenia, and agranulocytosis [3]."

citations = build_citations(answer_text, results, who_source_urls)
for c in citations:
    print(c)

c:\Users\DELL\Desktop\medrag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading sparse model 'Qdrant/bm25'...
Loading cross-encoder 'cross-encoder/ms-marco-MiniLM-L-6-v2'...
c:\Users\DELL\Desktop\medrag\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:06<00:00,  6.04s/it]

{'marker': 1, 'chunk_id': 'coronary artery disease+heart failure+hyperlipidemia+stroke_who_table_263', 'source': 'who', 'title': 'Prevention of cardiovascular disease: guidelines for assessment and management of total cardiovascular risk', 'url': None, 'linked_images': []}
{'marker': 3, 'chunk_id': 'Ramipril_openfda_6', 'source': 'openfda', 'title': 'Ramipril — FDA Label (Warnings And Cautions)', 'url': None, 'linked_images': []}


# Run the full citation build on a real answer

In [10]:
citations = build_citations(answer_text, results, who_source_urls)
for c in citations:
    print(c)

{'marker': 1, 'chunk_id': 'coronary artery disease+heart failure+hyperlipidemia+stroke_who_table_263', 'source': 'who', 'title': 'Prevention of cardiovascular disease: guidelines for assessment and management of total cardiovascular risk', 'url': None, 'linked_images': []}
{'marker': 3, 'chunk_id': 'Ramipril_openfda_6', 'source': 'openfda', 'title': 'Ramipril — FDA Label (Warnings And Cautions)', 'url': None, 'linked_images': []}


# Fix: split the combined source_id and look up any component topic

In [11]:
def get_who_source_url(source_id: str, who_source_urls: dict) -> str:
    """WHO chunks shared across topics have a combined source_id
    ('topic1+topic2+...'), but who_source_urls is keyed by individual
    topic filenames. Split on '+' and try each component - all topics
    in the group share the same underlying document/URL, so the first
    match is correct."""
    for topic in source_id.split("+"):
        if topic in who_source_urls:
            return who_source_urls[topic]
    return None


# patch get_display_info to use this instead of a direct dict lookup
def get_display_info(payload: dict, who_source_urls: dict) -> dict:
    source = payload["source"]

    if source == "who":
        title = payload["metadata"].get("title", payload.get("source_id", "WHO Guideline"))
        url = get_who_source_url(payload["source_id"], who_source_urls)
        return {"title": title, "url": url}

    if source == "openfda":
        drug_name = payload["source_id"]
        field = payload["metadata"].get("field", "")
        field_display = field.replace("_", " ").title()
        title = f"{drug_name} — FDA Label ({field_display})" if field else f"{drug_name} — FDA Label"
        return {"title": title, "url": None}

    if source == "pubmed":
        title = payload["metadata"].get("title", f"PubMed article {payload['source_id']}")
        pmid = payload["source_id"]
        return {"title": title, "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"}

    return {"title": "Unknown source", "url": None}


citations = build_citations(answer_text, results, who_source_urls)
for c in citations:
    print(c)

{'marker': 1, 'chunk_id': 'coronary artery disease+heart failure+hyperlipidemia+stroke_who_table_263', 'source': 'who', 'title': 'Prevention of cardiovascular disease: guidelines for assessment and management of total cardiovascular risk', 'url': 'https://iris.who.int/server/api/core/bitstreams/f106282d-01ad-4978-9a61-27fb1a8c305d/content', 'linked_images': []}
{'marker': 3, 'chunk_id': 'Ramipril_openfda_6', 'source': 'openfda', 'title': 'Ramipril — FDA Label (Warnings And Cautions)', 'url': None, 'linked_images': []}


# Test citation with a real linked image

In [12]:
# hypertension_who_text_16 was our confirmed Phase 7 test case with a real linked image
query = "hypertension medication side effects"
results_test = search_with_reranking(qdrant, query, candidate_pool_size=20, top_n=10)

# find if our known linked chunk shows up, or manually construct a test
for r in results_test:
    if r["payload"].get("linked_images"):
        print(f"chunk_id={r['chunk_id']}  linked_images={r['payload']['linked_images']}")

Batches: 100%|██████████| 1/1 [00:08<00:00,  8.42s/it]


# Directly test citation building on a known linked chunk

In [13]:
from medrag.processing.models import Chunk

# hypertension_who_text_16 is our confirmed Phase 7/8 test case
known_point_id = who_chunk_lookup["hypertension_who_text_16"].point_id if "who_chunk_lookup" in dir() else None

# fetch directly by point_id via Qdrant retrieve (bypasses search entirely)
from medrag.processing.storage import load_chunks
who_topics_for_lookup = [t for group in WHO_TOPIC_GROUPS for t in group]
target_chunk = None
for topic in who_topics_for_lookup:
    for chunk in load_chunks(source="who", topic=topic, output_dir=CHUNKS_DIR):
        if chunk.chunk_id == "hypertension_who_text_16":
            target_chunk = chunk
            break
    if target_chunk:
        break

print(f"Found chunk: {target_chunk.chunk_id}, point_id={target_chunk.point_id}")

fetched = qdrant.retrieve(collection_name="medrag_text", ids=[target_chunk.point_id])[0]
print(f"linked_images in Qdrant payload: {fetched.payload.get('linked_images')}")

# build a fake results list + answer text to test citation building on this one chunk
fake_results = [{"payload": fetched.payload, "chunk_id": fetched.payload["chunk_id"]}]
fake_answer = "This is discussed further [1]."

test_citations = build_citations(fake_answer, fake_results, who_source_urls)
for c in test_citations:
    print(c)

Found chunk: hypertension_who_text_16, point_id=b5ad6a29-6462-5f8b-a86f-f1388637df34
linked_images in Qdrant payload: [{'filename': 'hypertension_page2_img0.png', 'point_id': 'a8cb34f5-df8f-5d96-82cb-64f746b42fe2', 'figure_number': 1}]
{'marker': 1, 'chunk_id': 'hypertension_who_text_16', 'source': 'who', 'title': 'Guideline for the pharmacological treatment of hypertension in adults', 'url': 'https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content', 'linked_images': [{'filename': 'hypertension_page2_img0.png', 'point_id': 'a8cb34f5-df8f-5d96-82cb-64f746b42fe2', 'figure_number': 1}]}
